# Notebook 4 — Pseudo-4D Motion Recovery

---

## The Problem: MRI and Motion Don't Mix

Standard MRI is a **static imaging technique**. It assumes the patient is perfectly still during the entire acquisition. In practice, patients breathe, their heart beats, and they shift slightly — all of which corrupt the scan.

Capturing *motion itself* requires **4D MRI**: a sequence of 3D volumes acquired over time (x, y, z, time). This is expensive, slow, and requires specialized sequences not available in most clinical scanners.

But here's the thing: **motion leaves a trace in standard 3D k-space** — even in scans that weren't designed to capture it.

---

## Key Concept: The DC Component as a Motion Sensor

Recall from Notebook 3 that k-space is filled line by line over time. Each line has a timestamp. The **central column** (the DC component column) passes through the center of k-space and is sampled repeatedly throughout the acquisition.

The magnitude of the DC column reflects the **average brightness** of the entire image at that moment. When the patient breathes in, their chest expands, tissue moves, and the average brightness shifts. When they breathe out, it shifts back.

So the DC column, sampled over time, is a proxy signal for bulk motion — even though no one designed it to be.

```
DC column magnitude over time:

High → patient at end of inhale (chest expanded, tissue shifted up)
Low  → patient at end of exhale (chest contracted, tissue shifted down)
```

This is called a **motion surrogate** — a signal that correlates with motion without directly measuring it.

---

## Key Concept: Retrospective Gating

Once you have a motion surrogate, you can sort the k-space frames you already acquired into **motion phases** — groups that represent similar body positions.

Then you reconstruct each group separately. The result is multiple images, each representing a different phase of the motion cycle — assembled after the fact from a standard scan. This is called **retrospective gating** ("retrospective" because the sorting happens after acquisition, not during).

```
Standard 3D scan (one acquisition)
    → Extract motion surrogate from DC column
    → Sort k-space frames by surrogate value
    → Group into N motion phases
    → Reconstruct each phase separately
    → Result: N images at different motion states = pseudo-4D volume
```

---

## Why "Pseudo" 4D?

A true 4D scan measures the same anatomy at many time points directly. Pseudo-4D *infers* motion states from a single scan by sorting frames. The time resolution is limited by how many k-space frames were acquired — but you get motion information out of data that was never intended to provide it.

This technique is used clinically in **cardiac MRI** and **radiation therapy planning** to account for respiratory motion.

---

## Note on This Dataset

The fastMRI knee single-coil test set contains **static scans** — the patient was instructed to stay still. So there's no real respiratory motion to recover here.

Instead, we **simulate a time series** by treating each slice as a pseudo-time frame. This is purely for demonstration — it shows how the algorithm works mechanically, even if the motion signal here is just anatomical variation across slices rather than true temporal motion.

The same code applied to a cardiac or abdominal dynamic scan would recover real motion phases.

---

## Step 1 — Import Libraries

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from kode.io import load_kspace, root_sum_of_squares
from kode.pseudo4d import extract_motion_surrogate, sort_by_motion_phase

---

## Step 2 — Build a Simulated Time Series

We load 20 slices and stack them as if they were 20 time frames of the same slice. Each slice has slightly different anatomy — the algorithm treats these differences as if they were motion.

The output shape is `(time, coils, height, width)` — the standard format for a k-space time series.

In [ ]:
H5_FILE = '../data/singlecoil_test/file1000022.h5'

slices = []
for i in range(20):
    ks = load_kspace(H5_FILE, slice_idx=i)
    slices.append(ks)

# Stack along a new time axis
kspace_timeseries = np.stack(slices, axis=0)
print(f'Time series shape: {kspace_timeseries.shape}')
print(f'  → (time frames=20, coils={kspace_timeseries.shape[1]}, height={kspace_timeseries.shape[2]}, width={kspace_timeseries.shape[3]})')

---

## Step 3 — Extract the Motion Surrogate

For each time frame, we take the central column of k-space and average its magnitude across coils and k-space rows. This gives one number per frame — the surrogate signal.

Plot this signal and look for variation. In a real respiratory scan, you'd see a smooth oscillation. Here, you'll see variation driven by the anatomical differences between slices.

In [ ]:
surrogate = extract_motion_surrogate(kspace_timeseries)

print(f'Surrogate signal — one value per frame:')
print(np.round(surrogate, 2))

plt.figure(figsize=(10, 3))
plt.plot(surrogate, marker='o', color='steelblue', linewidth=1.5, markersize=5)
plt.xlabel('Frame index (simulated time)')
plt.ylabel('DC component magnitude')
plt.title('Motion Surrogate Signal — Extracted from K-Space DC Component')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/pseudo4d_surrogate.png', dpi=150)
plt.show()
print('Saved to results/pseudo4d_surrogate.png')

---

## Step 4 — Sort Into Motion Phases and Reconstruct

We sort the 20 frames by their surrogate value (low to high), split into 4 equal groups, average the k-space within each group, and reconstruct.

- **Phase 1** = frames with the lowest surrogate value (one extreme)
- **Phase 4** = frames with the highest surrogate value (other extreme)

In a real cardiac scan, these would correspond to end-diastole (heart relaxed) and end-systole (heart contracted).

In [ ]:
phases = sort_by_motion_phase(kspace_timeseries, n_phases=4)
print(f'Split into {len(phases)} motion phases')

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
phase_labels = [
    'Phase 1\n(min surrogate)',
    'Phase 2',
    'Phase 3',
    'Phase 4\n(max surrogate)',
]

for ax, phase_kspace, label in zip(axes, phases, phase_labels):
    image = root_sum_of_squares(phase_kspace)
    ax.imshow(image, cmap='gray')
    ax.set_title(label, fontsize=10)
    ax.axis('off')

plt.suptitle('Pseudo-4D — Motion Phases Recovered from K-Space Time Structure', fontsize=13)
plt.tight_layout()
plt.savefig('../results/pseudo4d_phases.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/pseudo4d_phases.png')

---

## What This Means

You recovered motion information from a scan that wasn't designed to capture motion.

The clinical implications:

- **Radiation therapy planning** — tumors near the lungs move with breathing. Pseudo-4D from a standard CT or MRI can map the full range of tumor motion without a dedicated 4D scan session.
- **Cardiac imaging** — heart position changes significantly between beats. Retrospective gating from standard scans reduces the need for specialized cardiac-gated sequences.
- **Motion correction** — rather than correcting motion artifacts after reconstruction, you can sort frames by motion state before reconstruction, avoiding the artifact entirely.

All of this comes from reading a signal that already exists in standard k-space data — it just hasn't been used.